In [1]:
# ============================================================
# PS S6E5 - exp_20260520_045_cat_shared001v2_features_optuna_search_min
# CatBoost shared001v2 features Optuna search
#
# Base:
#   042: exp_20260520_042_cat_shared001v2_features_min
#
# Purpose:
#   Fix 042 feature engineering / folds / original-row policy.
#   Search CatBoost hyperparameters only with Optuna.
#
# This notebook is NOT a formal submission/refit notebook.
# Main outputs:
#   - best_params_045.json
#   - optuna_trials_045.csv
#   - study_summary_045.json
#   - memo_draft.yml
#
# Forbidden:
#   - new feature engineering
#   - 039 LOG features
#   - Normalized_TyreLife direct/proxy use
#   - pseudo label / future endpoint proxy
# ============================================================


In [2]:
import os
import gc
import json
import random
import warnings
import pickle
from datetime import datetime

import optuna
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, log_loss

from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 300)


In [3]:
# ============================================================
# Config
# ============================================================

class CFG:
    EXP_ID = "exp_20260520_045_cat_shared001v2_features_optuna_search_min"
    COMPETITION = "PS S6E5 Predicting F1 Pit Stops"
    METRIC = "AUC"

    TARGET_CANDIDATES = ["PitNextLap", "PitNextLab"]
    TARGET = "PitNextLap"
    ID_COL = "id"
    DANGER_COL = "Normalized_TyreLife"

    # 045:
    # Base is 042.
    # Feature engineering, folds, original-row concat policy are fixed.
    # Optuna searches CatBoost hyperparameters only.
    # Keep 015 policy: drop only group-stat features whose group key is Race_Year.
    # Do NOT add 039 LOG features.
    # Do NOT drop Race_Year itself / IsOriginalData / TyreAgeRatio_str.
    DROP_GROUPSTAT_KEYWORDS = [
        "_by_Race_Year",
    ]

    COMP_PATHS = [
        "/kaggle/input/competitions/playground-series-s6e5",
        "/kaggle/input/playground-series-s6e5",
        ".",
    ]

    ORIGINAL_PATHS = [
        "/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv",
        "/kaggle/input/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv",
        "./f1_strategy_dataset_v4.csv",
    ]

    OUTDIR = Path(f"/kaggle/working/{EXP_ID}")

    SEED = 42
    N_SPLITS = 5
    USE_ORIGINAL = True

    # Optuna
    N_TRIALS = 40
    TIMEOUT_SEC = 60 * 60 * 10
    OPTUNA_DIRECTION = "maximize"

    # CatBoost fixed settings for search
    TASK_TYPE = "GPU"
    DEVICES = "0:1"

    ITERATIONS = 8000
    EARLY_STOPPING_ROUNDS = 300

    # Fixed GPU-safe bootstrap settings.
    # Do not search bagging_temperature when bootstrap_type="Bernoulli".
    BOOTSTRAP_TYPE = "Bernoulli"
    SUBSAMPLE = 0.8

    AUTO_CLASS_WEIGHTS = "Balanced"

    CLIP_LOW = 1e-6
    CLIP_HIGH = 1.0 - 1e-6

    # Keep Optuna output readable.
    VERBOSE = False

    # Baseline 042 references
    BASELINE_042_CV_AUC = 0.953240915414691
    BASELINE_042_PUBLIC_LB = 0.95283


CFG.OUTDIR.mkdir(parents=True, exist_ok=True)


In [4]:
# ============================================================
# Utilities
# ============================================================

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def print_section(title: str) -> None:
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)


def first_existing_dir(paths):
    for p in paths:
        path = Path(p)
        if (path / "train.csv").exists() and (path / "test.csv").exists():
            return path
    raise FileNotFoundError(f"No valid competition path found: {paths}")


def first_existing_file(paths, required=True):
    for p in paths:
        path = Path(p)
        if path.exists():
            return path
    if required:
        raise FileNotFoundError(f"No valid file path found: {paths}")
    return None


def resolve_target_column(df: pd.DataFrame) -> str:
    for c in CFG.TARGET_CANDIDATES:
        if c in df.columns:
            return c
    raise ValueError(f"No target column found among {CFG.TARGET_CANDIDATES}")


def reduce_mem_usage(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    for col in out.columns:
        dtype = out[col].dtype

        if pd.api.types.is_integer_dtype(dtype):
            c_min = out[col].min()
            c_max = out[col].max()

            if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                out[col] = out[col].astype(np.int8)
            elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                out[col] = out[col].astype(np.int16)
            elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                out[col] = out[col].astype(np.int32)

        elif pd.api.types.is_float_dtype(dtype):
            out[col] = pd.to_numeric(out[col], downcast="float")

    return out


def safe_divide(a, b, eps=1e-6):
    return a / (b + eps)


def clip_pred(x):
    return np.clip(x, CFG.CLIP_LOW, CFG.CLIP_HIGH)


seed_everything(CFG.SEED)

In [5]:
# ============================================================
# 015: Race_Year group-stat drop utility
# ============================================================

def is_race_year_groupstat_feature(col: str) -> bool:
    """
    Drop only features generated by add_light_group_stats
    with Race_Year as group key.

    Expected patterns:
      - LapTime_Delta_mean_by_Race_Year
      - LapTime_Delta_std_by_Race_Year
      - LapTime_Delta_diff_mean_by_Race_Year
      - Position_Change_mean_by_Race_Year
      - Position_Change_std_by_Race_Year
      - Position_Change_diff_mean_by_Race_Year
      - RaceProgress_mean_by_Race_Year
      - RaceProgress_std_by_Race_Year
      - RaceProgress_diff_mean_by_Race_Year
      - TyreLife_mean_by_Race_Year
      - TyreLife_std_by_Race_Year
      - TyreLife_diff_mean_by_Race_Year
    """
    return any(keyword in col for keyword in CFG.DROP_GROUPSTAT_KEYWORDS)


def drop_015_groupstat_features(feature_cols):
    """
    015 is a strict ablation from 012.
    Drop only Race_Year group-stat features.

    Do not drop:
      - Race_Year itself
      - Race_Year_count
      - Race_Year_freq
      - IsOriginalData
      - TyreAgeRatio_str
      - Race_Compound_Stint group stats
    """
    return [c for c in feature_cols if not is_race_year_groupstat_feature(c)]


def assert_015_feature_policy(feature_cols):
    """
    Guardrail for 015.
    This experiment is invalid if Race_Year group-stat features remain.
    It is also invalid if Race_Year itself or other key risk features are accidentally dropped.
    """
    remained = [c for c in feature_cols if is_race_year_groupstat_feature(c)]
    assert len(remained) == 0, (
        "015 policy violation: Race_Year group-stat features still remain: "
        + str(remained[:20])
    )

    assert "Race_Year" in feature_cols, (
        "015 policy violation: Race_Year itself was dropped. "
        "015 should drop only Race_Year group stats."
    )

    assert "IsOriginalData" in feature_cols, (
        "015 policy violation: IsOriginalData was dropped. "
        "015 should keep IsOriginalData."
    )

    assert "TyreAgeRatio_str" in feature_cols, (
        "015 policy violation: TyreAgeRatio_str was dropped. "
        "015 should keep TyreAgeRatio_str."
    )

In [6]:
# ============================================================
# Load data
# ============================================================

print_section("Load Data")

COMP_PATH = first_existing_dir(CFG.COMP_PATHS)

train_raw = pd.read_csv(COMP_PATH / "train.csv")
test_raw = pd.read_csv(COMP_PATH / "test.csv")
sample_submission = pd.read_csv(COMP_PATH / "sample_submission.csv")

target_col_train = resolve_target_column(train_raw)
if target_col_train != CFG.TARGET:
    train_raw = train_raw.rename(columns={target_col_train: CFG.TARGET})

target_col_sub = resolve_target_column(sample_submission)
if target_col_sub != CFG.TARGET:
    sample_submission = sample_submission.rename(columns={target_col_sub: CFG.TARGET})

original_raw = None
original_path = first_existing_file(CFG.ORIGINAL_PATHS, required=False)

if CFG.USE_ORIGINAL and original_path is not None:
    original_raw = pd.read_csv(original_path)

    if CFG.TARGET not in original_raw.columns:
        original_target = resolve_target_column(original_raw)
        original_raw = original_raw.rename(columns={original_target: CFG.TARGET})

    print("Loaded original:", original_path, original_raw.shape)

print("COMP_PATH:", COMP_PATH)
print("train_raw:", train_raw.shape)
print("test_raw :", test_raw.shape)
print("sample_submission:", sample_submission.shape)
print("target mean train:", train_raw[CFG.TARGET].mean())

assert CFG.ID_COL in train_raw.columns
assert CFG.ID_COL in test_raw.columns
assert CFG.ID_COL in sample_submission.columns
assert CFG.TARGET in train_raw.columns
assert CFG.TARGET in sample_submission.columns
assert test_raw[CFG.ID_COL].equals(sample_submission[CFG.ID_COL])

if original_raw is not None:
    assert CFG.TARGET in original_raw.columns
    print("target mean original:", original_raw[CFG.TARGET].mean())

    if CFG.DANGER_COL in original_raw.columns:
        original_raw = original_raw.drop(columns=[CFG.DANGER_COL])
        print(f"Dropped danger column from original: {CFG.DANGER_COL}")

train_raw = reduce_mem_usage(train_raw)
test_raw = reduce_mem_usage(test_raw)

if original_raw is not None:
    original_raw = reduce_mem_usage(original_raw)

train_ids = train_raw[CFG.ID_COL].copy()
test_ids = test_raw[CFG.ID_COL].copy()


Load Data
Loaded original: /kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv (101371, 16)
COMP_PATH: /kaggle/input/competitions/playground-series-s6e5
train_raw: (439140, 16)
test_raw : (188165, 15)
sample_submission: (188165, 2)
target mean train: 0.19898210137996994
target mean original: 0.25479673673930414
Dropped danger column from original: Normalized_TyreLife


In [7]:
# ============================================================
# shared_004 style feature engineering
# ============================================================

BASE_CAT_COLS = ["Driver", "Compound", "Race"]

NUMERIC_BASE_COLS = [
    "Year",
    "PitStop",
    "LapNumber",
    "Stint",
    "TyreLife",
    "Position",
    "LapTime (s)",
    "LapTime_Delta",
    "Cumulative_Degradation",
    "RaceProgress",
    "Position_Change",
]


def add_domain_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    eps = 1e-6

    for col in BASE_CAT_COLS:
        if col in out.columns:
            out[col] = out[col].astype("string").fillna("__MISSING__").astype(str)

    if {"LapNumber", "RaceProgress"}.issubset(out.columns):
        out["EstimatedTotalLaps"] = safe_divide(
            out["LapNumber"],
            out["RaceProgress"].clip(lower=eps),
            eps,
        ).replace([np.inf, -np.inf], np.nan).clip(1, 120)

        out["LapsRemaining"] = (
            out["EstimatedTotalLaps"] - out["LapNumber"]
        ).clip(lower=0)

        out["LapProgress_x_LapNumber"] = out["LapNumber"] * out["RaceProgress"]

    if {"TyreLife", "LapNumber"}.issubset(out.columns):
        out["TyreAgeRatio"] = safe_divide(out["TyreLife"], out["LapNumber"].clip(lower=1), eps)
        out["LapPerTyreLife"] = safe_divide(out["LapNumber"], out["TyreLife"] + 1, eps)
        out["LapMinusTyreLife"] = out["LapNumber"] - out["TyreLife"]
        out["TyreLifeMinusLap"] = out["TyreLife"] - out["LapNumber"]

    if {"TyreLife", "EstimatedTotalLaps"}.issubset(out.columns):
        out["TyreAgeVsRace"] = safe_divide(
            out["TyreLife"],
            out["EstimatedTotalLaps"].clip(lower=1),
            eps,
        )

    if {"TyreLife", "RaceProgress"}.issubset(out.columns):
        out["PitWindowPressure"] = out["TyreLife"] * out["RaceProgress"]
        out["TyreLife_x_RaceProgress"] = out["TyreLife"] * out["RaceProgress"]

    if {"TyreLife", "LapsRemaining"}.issubset(out.columns):
        out["TyreLife_to_LapsRemaining"] = safe_divide(out["TyreLife"], out["LapsRemaining"] + 1, eps)
        out["LapsRemaining_to_TyreLife"] = safe_divide(out["LapsRemaining"], out["TyreLife"] + 1, eps)

    if {"Cumulative_Degradation", "TyreLife"}.issubset(out.columns):
        out["DegPerTyreLap"] = safe_divide(
            out["Cumulative_Degradation"],
            out["TyreLife"].clip(lower=1),
            eps,
        )
        out["AbsDegPerTyreLap"] = safe_divide(
            out["Cumulative_Degradation"].abs(),
            out["TyreLife"].clip(lower=1),
            eps,
        )

    if {"Cumulative_Degradation", "LapNumber"}.issubset(out.columns):
        out["DegPerRaceLap"] = safe_divide(
            out["Cumulative_Degradation"],
            out["LapNumber"].clip(lower=1),
            eps,
        )

    if {"LapTime_Delta", "TyreLife"}.issubset(out.columns):
        out["DeltaPerTyreLap"] = safe_divide(
            out["LapTime_Delta"],
            out["TyreLife"].clip(lower=1),
            eps,
        )
        out["AbsDeltaPerTyreLap"] = safe_divide(
            out["LapTime_Delta"].abs(),
            out["TyreLife"].clip(lower=1),
            eps,
        )

    if "LapTime_Delta" in out.columns:
        out["DeltaAbs"] = out["LapTime_Delta"].abs()
        out["LapTimeDeltaPositive"] = (out["LapTime_Delta"] > 0).astype(np.int8)
        out["LapTimeDeltaNegative"] = (out["LapTime_Delta"] < 0).astype(np.int8)

    if "Cumulative_Degradation" in out.columns:
        out["Abs_Cumulative_Degradation"] = out["Cumulative_Degradation"].abs()
        out["Positive_Degradation"] = (out["Cumulative_Degradation"] > 0).astype(np.int8)

    if "Position_Change" in out.columns:
        out["Abs_Position_Change"] = out["Position_Change"].abs()
        out["Gained_Position"] = (out["Position_Change"] > 0).astype(np.int8)
        out["Lost_Position"] = (out["Position_Change"] < 0).astype(np.int8)

    if {"Position", "RaceProgress"}.issubset(out.columns):
        out["PositionPressure"] = out["Position"] * out["RaceProgress"]

    if {"Stint", "TyreLife"}.issubset(out.columns):
        out["StintPressure"] = out["Stint"] * out["TyreLife"]
        out["TyreLife_x_Stint"] = out["TyreLife"] * out["Stint"]

    if {"Stint", "LapNumber"}.issubset(out.columns):
        out["Stint_x_LapNumber"] = out["Stint"] * out["LapNumber"]
        out["Is_First_Stint"] = (out["Stint"] == 1).astype(np.int8)
        out["Is_Late_Stint"] = (out["Stint"] >= 3).astype(np.int8)

    if "RaceProgress" in out.columns:
        out["Early_Race"] = (out["RaceProgress"] <= 0.25).astype(np.int8)
        out["Mid_Race"] = (
            (out["RaceProgress"] > 0.25)
            & (out["RaceProgress"] <= 0.65)
        ).astype(np.int8)
        out["Late_Race"] = (out["RaceProgress"] > 0.65).astype(np.int8)

        out["RacePhase"] = pd.cut(
            out["RaceProgress"],
            bins=[-np.inf, 0.20, 0.40, 0.60, 0.80, np.inf],
            labels=["P1", "P2", "P3", "P4", "P5"],
        ).astype(str)

    if "LapNumber" in out.columns:
        out["LapBin"] = pd.cut(
            out["LapNumber"],
            bins=[-np.inf, 5, 10, 20, 35, 50, np.inf],
            labels=[
                "L_000_005",
                "L_006_010",
                "L_011_020",
                "L_021_035",
                "L_036_050",
                "L_051_plus",
            ],
        ).astype(str)

    if "TyreLife" in out.columns:
        out["TyreLifeBin"] = pd.cut(
            out["TyreLife"],
            bins=[-np.inf, 3, 7, 12, 20, 30, np.inf],
            labels=[
                "T_000_003",
                "T_004_007",
                "T_008_012",
                "T_013_020",
                "T_021_030",
                "T_031_plus",
            ],
        ).astype(str)

    if "Position" in out.columns:
        out["PositionBin"] = pd.cut(
            out["Position"],
            bins=[-np.inf, 3, 8, 14, np.inf],
            labels=["front", "upper_mid", "lower_mid", "back"],
        ).astype(str)

    def make_cross(name, cols):
        if set(cols).issubset(out.columns):
            val = out[cols[0]].astype(str)
            for c in cols[1:]:
                val = val + "_" + out[c].astype(str)
            out[name] = val

    make_cross("Race_Year", ["Race", "Year"])
    make_cross("Compound_Stint", ["Compound", "Stint"])
    make_cross("Driver_Race", ["Driver", "Race"])
    make_cross("Driver_Compound", ["Driver", "Compound"])
    make_cross("Race_Compound", ["Race", "Compound"])
    make_cross("Race_Compound_Stint", ["Race", "Compound", "Stint"])
    make_cross("Compound_RacePhase", ["Compound", "RacePhase"])
    make_cross("Compound_TyreLifeBin", ["Compound", "TyreLifeBin"])
    make_cross("RacePhase_TyreLifeBin", ["RacePhase", "TyreLifeBin"])

    out = out.replace([np.inf, -np.inf], np.nan)

    for col in out.select_dtypes(include=["float64"]).columns:
        out[col] = out[col].astype(np.float32)

    return out


def add_digit_features(df: pd.DataFrame, numeric_cols=None, int_digit_limit=3, decimal_digit_limit=2) -> pd.DataFrame:
    out = df.copy()

    if numeric_cols is None:
        numeric_cols = out.select_dtypes(include=[np.number]).columns.tolist()

    for col in numeric_cols:
        if col not in out.columns:
            continue

        s = out[col].fillna(0).astype(float)
        abs_s = s.abs()

        for i in range(int_digit_limit):
            new_col = f"{col}_int_digit_{i + 1}"
            out[new_col] = ((abs_s // (10 ** i)) % 10).astype(np.int8)

        if pd.api.types.is_float_dtype(out[col]):
            for i in range(1, decimal_digit_limit + 1):
                new_col = f"{col}_dec_digit_{i}"
                out[new_col] = ((abs_s * (10 ** i)).round().astype(int) % 10).astype(np.int8)

    return out


def add_float_signature_features(df: pd.DataFrame, float_cols=None) -> pd.DataFrame:
    out = df.copy()

    selected = [
        "RaceProgress",
        "LapTime (s)",
        "LapTime_Delta",
        "Cumulative_Degradation",
        "TyreAgeRatio",
        "DegPerTyreLap",
        "DegPerRaceLap",
        "DeltaPerTyreLap",
        "DeltaAbs",
        "PitWindowPressure",
        "EstimatedTotalLaps",
        "LapsRemaining",
        "LapMinusTyreLife",
    ]

    float_cols = [c for c in selected if c in out.columns]

    for col in float_cols:
        scaled = (out[col].fillna(0).astype(float) * 100).round().astype(int).abs()

        for i in range(5):
            new_col = f"{col}_sig_{i + 1}"
            digit = ((scaled // (10 ** i)) % 10).astype(np.int8)

            if digit.nunique() > 1:
                out[new_col] = digit.astype(str)

    return out


def add_string_precision_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    if "RaceProgress" in out.columns:
        out["RaceProgress_str"] = out["RaceProgress"].round(4).astype(str)

    if "EstimatedTotalLaps" in out.columns:
        out["EstimatedTotalLaps_str"] = out["EstimatedTotalLaps"].round(1).astype(str)

    if "TyreAgeRatio" in out.columns:
        out["TyreAgeRatio_str"] = out["TyreAgeRatio"].round(3).astype(str)

    return out


# ============================================================
# 042: shared001v2-style features from 029/038 RealMLP branch
# ============================================================

SHARED001V2_042_NUMERIC_SOURCE_COLS = [
    "Year",
    "PitStop",
    "LapNumber",
    "Stint",
    "TyreLife",
    "Position",
    "LapTime (s)",
    "LapTime_Delta",
    "Cumulative_Degradation",
    "RaceProgress",
    "Position_Change",
]

SHARED001V2_042_CAT_FEATURES = [
    "PitStop_cat_",
    "Race_Year_",
    "Race_Compound_",
    "RaceProgress_200_quantile_bin_",
    "LapTime (s)_7_quantile_bin_",
]

SHARED001V2_042_COUNT_FEATURES = [
    "_PitStop_cat_count",
    "_Race_Year_count",
    "_Race_Compound_count",
    "_RaceProgress_200_quantile_bin_count",
    "_LapTime_s_7_quantile_bin_count",
]


def _to_safe_str_series(s: pd.Series) -> pd.Series:
    return s.astype("string").fillna("__MISSING__").astype(str)


def _floor_cat_series(s: pd.Series) -> pd.Series:
    x = pd.to_numeric(s, errors="coerce")
    # Use a dedicated missing token before flooring to avoid silent imputation.
    floored = np.floor(x).astype("float")
    return floored.astype("Int64").astype("string").fillna("__MISSING__").astype(str)


def add_shared001v2_style_features_042(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add 029/038 RealMLP-style categorical/bin features to CatBoost.
    These are row-wise transformations only; no target is used here.

    Important:
    - 039 LOG features are intentionally not added.
    - Normalized_TyreLife and any endpoint proxy are not reconstructed.
    """
    out = df.copy()

    if "PitStop" in out.columns:
        out["PitStop_cat_"] = _floor_cat_series(out["PitStop"])

    if {"Race", "Year"}.issubset(out.columns):
        out["Race_Year_"] = _to_safe_str_series(out["Race"]) + "_" + _to_safe_str_series(out["Year"])

    if {"Race", "Compound"}.issubset(out.columns):
        out["Race_Compound_"] = _to_safe_str_series(out["Race"]) + "_" + _to_safe_str_series(out["Compound"])

    # 029/038-style quantile bins.
    # qcut is avoided because train/test/original must receive the same deterministic mapping.
    if "RaceProgress" in out.columns:
        rp = pd.to_numeric(out["RaceProgress"], errors="coerce").clip(0, 1)
        bin_idx = np.floor(rp.fillna(-1) * 200).astype(int).clip(-1, 199)
        out["RaceProgress_200_quantile_bin_"] = np.where(
            bin_idx < 0,
            "__MISSING__",
            "rp200_" + pd.Series(bin_idx, index=out.index).astype(str),
        )

    if "LapTime (s)" in out.columns:
        lt = pd.to_numeric(out["LapTime (s)"], errors="coerce")
        # Use deterministic absolute bins rather than qcut.
        # qcut per-frame would make train/test/original bin definitions inconsistent.
        codes = pd.cut(
            lt,
            bins=[-np.inf, 70, 75, 80, 85, 90, 95, np.inf],
            labels=False,
        )
        out["LapTime (s)_7_quantile_bin_"] = (
            "lt7_" + codes.astype("Int64").astype("string")
        ).fillna("__MISSING__").astype(str)

    for col in SHARED001V2_042_NUMERIC_SOURCE_COLS:
        if col in out.columns:
            out[f"{col}_floor_cat_042"] = _floor_cat_series(out[col])

    return out


def add_shared001v2_counts_042(train_df, test_df, original_df=None):
    """
    Count-encode only the newly added shared001v2-style categorical/bin columns.
    Counts are calculated from train + test + original feature space, without target use.
    """
    count_source_cols = [
        "PitStop_cat_",
        "Race_Year_",
        "Race_Compound_",
        "RaceProgress_200_quantile_bin_",
        "LapTime (s)_7_quantile_bin_",
    ] + [
        f"{c}_floor_cat_042"
        for c in SHARED001V2_042_NUMERIC_SOURCE_COLS
        if f"{c}_floor_cat_042" in train_df.columns and f"{c}_floor_cat_042" in test_df.columns
    ]

    count_source_cols = [
        c for c in count_source_cols
        if c in train_df.columns and c in test_df.columns
    ]

    frames = [train_df, test_df]
    if original_df is not None:
        frames.append(original_df)

    base = pd.concat(
        [f[count_source_cols].astype(str) for f in frames],
        axis=0,
        ignore_index=True,
    )
    total = len(base)

    for col in count_source_cols:
        counts = base[col].astype(str).value_counts(dropna=False)
        safe_name = col.replace(" ", "_").replace("(", "").replace(")", "")
        for f in frames:
            f[f"_{safe_name}_count"] = f[col].astype(str).map(counts).fillna(0).astype(np.float32)
            f[f"_{safe_name}_freq"] = (f[f"_{safe_name}_count"] / total).astype(np.float32)

    del base
    gc.collect()

    if original_df is not None:
        return train_df, test_df, original_df

    return train_df, test_df, None



def add_frequency_features(train_df, test_df, original_df=None):
    freq_cols = [
        "Driver",
        "Race",
        "Compound",
        "Race_Year",
        "Compound_Stint",
        "Driver_Race",
        "Driver_Compound",
        "Race_Compound",
        "Race_Compound_Stint",
        "Compound_RacePhase",
        "Compound_TyreLifeBin",
        "RacePhase_TyreLifeBin",
        "LapBin",
        "TyreLifeBin",
        "PositionBin",
    ]

    freq_cols = [c for c in freq_cols if c in train_df.columns and c in test_df.columns]

    frames = [train_df, test_df]
    if original_df is not None:
        frames.append(original_df)

    base = pd.concat([f[freq_cols] for f in frames], axis=0, ignore_index=True)
    total = len(base)

    for col in freq_cols:
        counts = base[col].astype(str).value_counts(dropna=False)

        for f in frames:
            f[f"{col}_count"] = f[col].astype(str).map(counts).fillna(0).astype(np.float32)
            f[f"{col}_freq"] = (f[f"{col}_count"] / total).astype(np.float32)

    del base
    gc.collect()

    if original_df is not None:
        return train_df, test_df, original_df

    return train_df, test_df, None


def add_light_group_stats(train_df, test_df, original_df=None):
    group_cols_list = [
        ["Race_Year"],
        ["Race_Compound_Stint"],
        ["Driver_Race"],
        ["Compound_Stint"],
    ]

    value_cols = [
        "LapTime_Delta",
        "Position_Change",
        "RaceProgress",
        "TyreLife",
    ]

    frames = [train_df, test_df]
    if original_df is not None:
        frames.append(original_df)

    for group_cols in group_cols_list:
        group_cols = [c for c in group_cols if all(c in f.columns for f in frames)]

        if not group_cols:
            continue

        group_name = "_".join(group_cols)

        for value_col in value_cols:
            if not all(value_col in f.columns for f in frames):
                continue

            base = pd.concat(
                [f[group_cols + [value_col]] for f in frames],
                axis=0,
                ignore_index=True,
            )

            stats = (
                base.groupby(group_cols, dropna=False)[value_col]
                .agg(["mean", "std"])
                .reset_index()
            )

            stats.columns = group_cols + [
                f"{value_col}_mean_by_{group_name}",
                f"{value_col}_std_by_{group_name}",
            ]

            for idx, f in enumerate(frames):
                frames[idx] = f.merge(stats, on=group_cols, how="left")

                mean_col = f"{value_col}_mean_by_{group_name}"
                frames[idx][f"{value_col}_diff_mean_by_{group_name}"] = (
                    frames[idx][value_col] - frames[idx][mean_col]
                ).astype(np.float32)

            del base, stats
            gc.collect()

    if original_df is not None:
        return frames[0], frames[1], frames[2]

    return frames[0], frames[1], None


def clean_columns(train_df, test_df, original_df=None):
    train_df = train_df.copy()
    test_df = test_df.copy()

    if original_df is not None:
        original_df = original_df.copy()

    common_cols = [c for c in train_df.columns if c in test_df.columns]

    train_df = train_df[common_cols + [CFG.TARGET]]
    test_df = test_df[common_cols]

    if original_df is not None:
        for c in common_cols:
            if c not in original_df.columns:
                original_df[c] = np.nan

        original_df = original_df[common_cols + [CFG.TARGET]]

    return train_df, test_df, original_df


def fill_missing_and_types(train_df, test_df, original_df=None):
    frames = [train_df, test_df]
    if original_df is not None:
        frames.append(original_df)

    all_feature_cols = [c for c in train_df.columns if c != CFG.TARGET]

    cat_cols = []

    for col in all_feature_cols:
        if any(
            (
                f[col].dtype == "object"
                or str(f[col].dtype).startswith("category")
                or str(f[col].dtype).startswith("string")
            )
            for f in frames
            if col in f.columns
        ):
            cat_cols.append(col)

    num_cols = [c for c in all_feature_cols if c not in cat_cols]

    for col in cat_cols:
        values = pd.concat(
            [f[col].astype("string") for f in frames if col in f.columns],
            axis=0,
        )
        mode_values = values.mode()
        mode_value = mode_values.iloc[0] if len(mode_values) else "__MISSING__"

        for f in frames:
            if col in f.columns:
                f[col] = f[col].astype("string").fillna(mode_value).astype(str)

    for col in num_cols:
        values = pd.concat([f[col] for f in frames if col in f.columns], axis=0)
        fill_value = values.median()

        for f in frames:
            if col in f.columns:
                f[col] = f[col].replace([np.inf, -np.inf], np.nan).fillna(fill_value)
                if f[col].dtype == "float64":
                    f[col] = f[col].astype(np.float32)

    train_df = reduce_mem_usage(train_df)
    test_df = reduce_mem_usage(test_df)

    if original_df is not None:
        original_df = reduce_mem_usage(original_df)

    return train_df, test_df, original_df, cat_cols

In [8]:
# ============================================================
# Feature Engineering
# ============================================================

print_section("Feature Engineering")

train_fe = train_raw.copy()
test_fe = test_raw.copy()

if original_raw is not None and CFG.TARGET in original_raw.columns:
    original_fe = original_raw.copy()
else:
    original_fe = None

if original_fe is not None and CFG.DANGER_COL in original_fe.columns:
    original_fe = original_fe.drop(columns=[CFG.DANGER_COL])
    print(f"Dropped danger column from original_fe: {CFG.DANGER_COL}")

train_fe["IsOriginalData"] = 0
test_fe["IsOriginalData"] = 0

if original_fe is not None:
    original_fe["IsOriginalData"] = 1

train_fe = add_domain_features(train_fe)
test_fe = add_domain_features(test_fe)

if original_fe is not None:
    original_fe = add_domain_features(original_fe)

digit_source_cols = [
    "Year",
    "PitStop",
    "LapNumber",
    "Stint",
    "TyreLife",
    "Position",
    "LapTime (s)",
    "LapTime_Delta",
    "Cumulative_Degradation",
    "RaceProgress",
    "Position_Change",
    "EstimatedTotalLaps",
    "LapsRemaining",
    "TyreAgeRatio",
    "DegPerTyreLap",
    "DegPerRaceLap",
    "DeltaPerTyreLap",
    "DeltaAbs",
    "PositionPressure",
    "StintPressure",
    "PitWindowPressure",
    "LapMinusTyreLife",
]
digit_source_cols = [c for c in digit_source_cols if c in train_fe.columns and c in test_fe.columns]

train_fe = add_digit_features(train_fe, digit_source_cols)
test_fe = add_digit_features(test_fe, digit_source_cols)

if original_fe is not None:
    original_fe = add_digit_features(original_fe, digit_source_cols)

train_fe = add_float_signature_features(train_fe)
test_fe = add_float_signature_features(test_fe)

if original_fe is not None:
    original_fe = add_float_signature_features(original_fe)

train_fe = add_string_precision_features(train_fe)
test_fe = add_string_precision_features(test_fe)

if original_fe is not None:
    original_fe = add_string_precision_features(original_fe)

# 042: add shared001v2-style categorical/bin/count features.
# Keep this before fill_missing/types so CatBoost receives these as categorical columns.
train_fe = add_shared001v2_style_features_042(train_fe)
test_fe = add_shared001v2_style_features_042(test_fe)

if original_fe is not None:
    original_fe = add_shared001v2_style_features_042(original_fe)

train_fe, test_fe, original_fe = add_frequency_features(train_fe, test_fe, original_fe)
train_fe, test_fe, original_fe = add_shared001v2_counts_042(train_fe, test_fe, original_fe)
train_fe, test_fe, original_fe = add_light_group_stats(train_fe, test_fe, original_fe)

train_fe, test_fe, original_fe = clean_columns(train_fe, test_fe, original_fe)
train_fe, test_fe, original_fe, cat_cols = fill_missing_and_types(train_fe, test_fe, original_fe)

print("train_fe:", train_fe.shape)
print("test_fe :", test_fe.shape)

if original_fe is not None:
    print("original_fe:", original_fe.shape)

print("categorical features:", len(cat_cols))


Feature Engineering
train_fe: (439140, 351)
test_fe : (188165, 350)
original_fe: (101371, 351)
categorical features: 93


In [9]:
# ============================================================
# Prepare matrices
# ============================================================

print_section("Prepare Matrices")

X_comp = train_fe.drop(columns=[CFG.TARGET, CFG.ID_COL], errors="ignore")
y_comp = train_fe[CFG.TARGET].astype(int).reset_index(drop=True)

X_test_final = test_fe.drop(columns=[CFG.ID_COL], errors="ignore")

if original_fe is not None:
    X_orig = original_fe.drop(columns=[CFG.TARGET, CFG.ID_COL], errors="ignore")
    y_orig = original_fe[CFG.TARGET].astype(int).reset_index(drop=True)
else:
    X_orig = None
    y_orig = None

common_features = [c for c in X_comp.columns if c in X_test_final.columns]

# 015差分: Race_Year group-stat featuresだけを除外
dropped_groupstat_features_015 = [
    c for c in common_features
    if is_race_year_groupstat_feature(c)
]

common_features = drop_015_groupstat_features(common_features)
assert_015_feature_policy(common_features)

print("045/042-base dropped Race_Year group-stat features:", dropped_groupstat_features_015)
print("045/042-base dropped feature count:", len(dropped_groupstat_features_015))
print("final feature count after 045/042-base drop:", len(common_features))
print("Race_Year in features:", "Race_Year" in common_features)
print("IsOriginalData in features:", "IsOriginalData" in common_features)
print("TyreAgeRatio_str in features:", "TyreAgeRatio_str" in common_features)
print(
    "Remaining Race_Year group-stat features:",
    [c for c in common_features if is_race_year_groupstat_feature(c)]
)

# 042 guardrails: these shared001v2-style columns should exist unless source columns are missing.
expected_shared001v2_features_042 = [
    c for c in [
        "PitStop_cat_",
        "Race_Year_",
        "Race_Compound_",
        "RaceProgress_200_quantile_bin_",
        "LapTime (s)_7_quantile_bin_",
    ]
    if c in X_comp.columns and c in X_test_final.columns
]
print("042 shared001v2-style base features:", expected_shared001v2_features_042)
print("042 shared001v2-style feature count:", len([
    c for c in common_features
    if c.endswith("_floor_cat_042")
    or c in expected_shared001v2_features_042
    or c.startswith("_PitStop_cat_")
    or c.startswith("_Race_Year_")
    or c.startswith("_Race_Compound_")
    or c.startswith("_RaceProgress_200_quantile_bin_")
    or c.startswith("_LapTime_s_7_quantile_bin_")
]))

X_comp = X_comp[common_features].reset_index(drop=True)
X_test_final = X_test_final[common_features].reset_index(drop=True)

if X_orig is not None:
    for c in common_features:
        if c not in X_orig.columns:
            X_orig[c] = np.nan
    X_orig = X_orig[common_features].reset_index(drop=True)

cat_cols = [c for c in cat_cols if c in common_features]
cat_features = [X_comp.columns.get_loc(c) for c in cat_cols]



Prepare Matrices
045/042-base dropped Race_Year group-stat features: ['LapTime_Delta_mean_by_Race_Year', 'LapTime_Delta_std_by_Race_Year', 'LapTime_Delta_diff_mean_by_Race_Year', 'Position_Change_mean_by_Race_Year', 'Position_Change_std_by_Race_Year', 'Position_Change_diff_mean_by_Race_Year', 'RaceProgress_mean_by_Race_Year', 'RaceProgress_std_by_Race_Year', 'RaceProgress_diff_mean_by_Race_Year', 'TyreLife_mean_by_Race_Year', 'TyreLife_std_by_Race_Year', 'TyreLife_diff_mean_by_Race_Year']
045/042-base dropped feature count: 12
final feature count after 045/042-base drop: 337
Race_Year in features: True
IsOriginalData in features: True
TyreAgeRatio_str in features: True
Remaining Race_Year group-stat features: []
042 shared001v2-style base features: ['PitStop_cat_', 'Race_Year_', 'Race_Compound_', 'RaceProgress_200_quantile_bin_', 'LapTime (s)_7_quantile_bin_']
042 shared001v2-style feature count: 26


In [10]:
# ============================================================
# CatBoost Optuna model functions
# ============================================================

def suggest_catboost_params(trial: optuna.trial.Trial) -> dict:
    """Search only CatBoost hyperparameters. Feature engineering is fixed to 042."""
    return {
        "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.08, log=True),
        "depth": trial.suggest_int("depth", 4, 8),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 30.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 0.0, 2.0),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 40),
        "border_count": trial.suggest_categorical("border_count", [128, 254]),
    }


def get_catboost_params(seed: int, task_type: str, trial_params: dict) -> dict:
    params = {
        "iterations": CFG.ITERATIONS,
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "auto_class_weights": CFG.AUTO_CLASS_WEIGHTS,

        # searched
        "learning_rate": trial_params["learning_rate"],
        "depth": trial_params["depth"],
        "l2_leaf_reg": trial_params["l2_leaf_reg"],
        "random_strength": trial_params["random_strength"],
        "min_data_in_leaf": trial_params["min_data_in_leaf"],
        "border_count": trial_params["border_count"],

        # fixed
        "bootstrap_type": CFG.BOOTSTRAP_TYPE,
        "subsample": CFG.SUBSAMPLE,

        "task_type": task_type,
        "random_seed": seed,
        "early_stopping_rounds": CFG.EARLY_STOPPING_ROUNDS,
        "allow_writing_files": False,
        "verbose": CFG.VERBOSE,
    }

    if task_type == "GPU":
        params["devices"] = CFG.DEVICES
    else:
        params["thread_count"] = -1

    return params


def fit_catboost_with_fallback(X_tr, y_tr, X_va, y_va, seed: int, trial_params: dict):
    params = get_catboost_params(seed=seed, task_type=CFG.TASK_TYPE, trial_params=trial_params)
    model = CatBoostClassifier(**params)

    try:
        model.fit(
            X_tr,
            y_tr,
            eval_set=(X_va, y_va),
            cat_features=cat_features,
            use_best_model=True,
        )
        return model, CFG.TASK_TYPE

    except Exception as e:
        print("CatBoost GPU/device training failed. Retrying with CPU.")
        print("Error:", repr(e))

        cpu_params = get_catboost_params(seed=seed, task_type="CPU", trial_params=trial_params)
        cpu_model = CatBoostClassifier(**cpu_params)

        cpu_model.fit(
            X_tr,
            y_tr,
            eval_set=(X_va, y_va),
            cat_features=cat_features,
            use_best_model=True,
        )

        return cpu_model, "CPU"


In [11]:
# ============================================================
# Optuna search
# ============================================================

print_section("Optuna Search")

folds = list(
    StratifiedKFold(
        n_splits=CFG.N_SPLITS,
        shuffle=True,
        random_state=CFG.SEED,
    ).split(X_comp, y_comp)
)

trial_records = []


def objective(trial: optuna.trial.Trial) -> float:
    trial_params = suggest_catboost_params(trial)

    oof_trial = np.zeros(len(X_comp), dtype=np.float32)
    fold_aucs = []
    fold_loglosses = []
    fold_best_iterations = []
    used_devices = []

    print("\n" + "=" * 90)
    print(f"Trial {trial.number}")
    print(trial_params)
    print("=" * 90)

    for fold, (tr_idx, va_idx) in enumerate(folds, 1):
        X_tr_comp = X_comp.iloc[tr_idx].copy().reset_index(drop=True)
        y_tr_comp = y_comp.iloc[tr_idx].copy().reset_index(drop=True)

        X_va = X_comp.iloc[va_idx].copy().reset_index(drop=True)
        y_va = y_comp.iloc[va_idx].copy().reset_index(drop=True)

        if X_orig is not None and CFG.USE_ORIGINAL:
            X_tr = pd.concat([X_tr_comp, X_orig], axis=0, ignore_index=True)
            y_tr = pd.concat([y_tr_comp, y_orig], axis=0, ignore_index=True)
        else:
            X_tr = X_tr_comp
            y_tr = y_tr_comp

        model, used_device = fit_catboost_with_fallback(
            X_tr=X_tr,
            y_tr=y_tr,
            X_va=X_va,
            y_va=y_va,
            seed=CFG.SEED + fold,
            trial_params=trial_params,
        )

        va_pred = clip_pred(model.predict_proba(X_va)[:, 1]).astype(np.float32)
        oof_trial[va_idx] = va_pred

        fold_auc = roc_auc_score(y_va, va_pred)
        fold_logloss = log_loss(y_va, va_pred)
        best_iter = int(model.get_best_iteration() or -1)

        fold_aucs.append(float(fold_auc))
        fold_loglosses.append(float(fold_logloss))
        fold_best_iterations.append(best_iter)
        used_devices.append(used_device)

        print(
            f"Trial {trial.number:03d} Fold {fold}: "
            f"AUC={fold_auc:.9f}, LogLoss={fold_logloss:.9f}, "
            f"best_iter={best_iter}, device={used_device}"
        )

        del model, X_tr_comp, y_tr_comp, X_va, y_va, X_tr, y_tr, va_pred
        gc.collect()

    cv_auc = float(roc_auc_score(y_comp, oof_trial))
    cv_logloss = float(log_loss(y_comp, clip_pred(oof_trial)))

    trial.set_user_attr("cv_auc", cv_auc)
    trial.set_user_attr("cv_logloss", cv_logloss)
    trial.set_user_attr("fold_aucs", fold_aucs)
    trial.set_user_attr("fold_loglosses", fold_loglosses)
    trial.set_user_attr("fold_best_iterations", fold_best_iterations)
    trial.set_user_attr("used_devices", sorted(set(used_devices)))

    record = {
        "trial": int(trial.number),
        "cv_auc": cv_auc,
        "cv_logloss": cv_logloss,
        "fold_auc_mean": float(np.mean(fold_aucs)),
        "fold_auc_std": float(np.std(fold_aucs, ddof=1)),
        "fold_logloss_mean": float(np.mean(fold_loglosses)),
        "fold_logloss_std": float(np.std(fold_loglosses, ddof=1)),
        "best_iteration_mean": float(np.mean(fold_best_iterations)),
        "best_iteration_min": int(np.min(fold_best_iterations)),
        "best_iteration_max": int(np.max(fold_best_iterations)),
        "used_devices": ",".join(sorted(set(used_devices))),
        **trial_params,
    }
    trial_records.append(record)

    print(
        f"Trial {trial.number:03d} CV AUC={cv_auc:.9f}, "
        f"CV LogLoss={cv_logloss:.9f}, "
        f"delta_vs_042={cv_auc - CFG.BASELINE_042_CV_AUC:+.9f}"
    )

    return cv_auc


sampler = optuna.samplers.TPESampler(seed=CFG.SEED)
study = optuna.create_study(direction=CFG.OPTUNA_DIRECTION, sampler=sampler)

study.optimize(
    objective,
    n_trials=CFG.N_TRIALS,
    timeout=CFG.TIMEOUT_SEC,
    gc_after_trial=True,
)

print("Best trial:", study.best_trial.number)
print("Best value:", study.best_value)
print("Best params:", study.best_params)
print("Delta vs 042:", study.best_value - CFG.BASELINE_042_CV_AUC)


[I 2026-05-20 10:47:05,996] A new study created in memory with name: no-name-a9a5326a-a60a-4fc1-a5f6-4dbca82a933f



Optuna Search

Trial 0
{'learning_rate': 0.028078987855609694, 'depth': 8, 'l2_leaf_reg': 12.057126287443763, 'random_strength': 1.1973169683940732, 'min_data_in_leaf': 7, 'border_count': 128}


Default metric period is 5 because AUC is/are not implemented for GPU


Trial 000 Fold 1: AUC=0.954291005, LogLoss=0.260741970, best_iter=4676, device=GPU


Default metric period is 5 because AUC is/are not implemented for GPU


...

Trial 006 Fold 3: AUC=0.953443236, LogLoss=0.261346304, best_iter=7988, device=GPU


Default metric period is 5 because AUC is/are not implemented for GPU


Trial 006 Fold 4: AUC=0.952615785, LogLoss=0.262158838, best_iter=7390, device=GPU


Default metric period is 5 because AUC is/are not implemented for GPU
[I 2026-05-20 22:44:24,420] Trial 6 finished with value: 0.953352253739158 and parameters: {'learning_rate': 0.015888820918127773, 'depth': 8, 'l2_leaf_reg': 2.411289811529197, 'random_strength': 1.325044568707964, 'min_data_in_leaf': 13, 'border_count': 254}. Best is trial 6 with value: 0.953352253739158.


Trial 006 Fold 5: AUC=0.953889870, LogLoss=0.256574241, best_iter=7980, device=GPU
Trial 006 CV AUC=0.953352254, CV LogLoss=0.260492538, delta_vs_042=+0.000111338
Best trial: 6
Best value: 0.953352253739158
Best params: {'learning_rate': 0.015888820918127773, 'depth': 8, 'l2_leaf_reg': 2.411289811529197, 'random_strength': 1.325044568707964, 'min_data_in_leaf': 13, 'border_count': 254}
Delta vs 042: 0.00011133832446696967


In [12]:
# ============================================================
# Optuna result
# ============================================================

print_section("Optuna Result")

trials_df = pd.DataFrame(trial_records).sort_values("cv_auc", ascending=False).reset_index(drop=True)

best_trial = study.best_trial
best_params = dict(study.best_params)
best_cv_auc = float(study.best_value)

summary = {
    "experiment_id": CFG.EXP_ID,
    "type": "optuna_search",
    "competition": CFG.COMPETITION,
    "metric": CFG.METRIC,
    "base_experiment": "exp_20260520_042_cat_shared001v2_features_min",
    "baseline_042_cv_auc": CFG.BASELINE_042_CV_AUC,
    "baseline_042_public_lb": CFG.BASELINE_042_PUBLIC_LB,
    "best_trial_number": int(best_trial.number),
    "best_cv_auc": best_cv_auc,
    "delta_vs_042_cv": best_cv_auc - CFG.BASELINE_042_CV_AUC,
    "best_params": best_params,
    "n_trials_requested": CFG.N_TRIALS,
    "n_trials_completed": len(study.trials),
    "timeout_sec": CFG.TIMEOUT_SEC,
    "feature_set": "same_as_042",
    "folds": CFG.N_SPLITS,
    "use_original": CFG.USE_ORIGINAL,
    "forbidden": [
        "Normalized_TyreLife",
        "Normalized_TyreLife proxy",
        "future endpoint proxy",
        "pseudo label",
        "new feature engineering",
        "039 LOG features",
    ],
}

print(json.dumps(summary, indent=2))
print("\nTop trials:")
display(trials_df.head(20))



Optuna Result
{
  "experiment_id": "exp_20260520_045_cat_shared001v2_features_optuna_search_min",
  "type": "optuna_search",
  "competition": "PS S6E5 Predicting F1 Pit Stops",
  "metric": "AUC",
  "base_experiment": "exp_20260520_042_cat_shared001v2_features_min",
  "baseline_042_cv_auc": 0.953240915414691,
  "baseline_042_public_lb": 0.95283,
  "best_trial_number": 6,
  "best_cv_auc": 0.953352253739158,
  "delta_vs_042_cv": 0.00011133832446696967,
  "best_params": {
    "learning_rate": 0.015888820918127773,
    "depth": 8,
    "l2_leaf_reg": 2.411289811529197,
    "random_strength": 1.325044568707964,
    "min_data_in_leaf": 13,
    "border_count": 254
  },
  "n_trials_requested": 40,
  "n_trials_completed": 7,
  "timeout_sec": 36000,
  "feature_set": "same_as_042",
  "folds": 5,
  "use_original": true,
  "forbidden": [
    "Normalized_TyreLife",
    "Normalized_TyreLife proxy",
    "future endpoint proxy",
    "pseudo label",
    "new feature engineering",
    "039 LOG features"
 

,trial,cv_auc,cv_logloss,fold_auc_mean,fold_auc_std,fold_logloss_mean,fold_logloss_std,best_iteration_mean,best_iteration_min,best_iteration_max,used_devices,learning_rate,depth,l2_leaf_reg,random_strength,min_data_in_leaf,border_count
0,6,0.953352,0.260493,0.953360,0.000774,0.260493,0.002301,7796.0,7390,7988,GPU,0.015889,8,2.411290,1.325045,13,254
1,0,0.953305,0.260984,0.953311,0.000849,0.260984,0.002006,4765.8,4561,5245,GPU,0.028079,8,12.057126,1.197317,7,128
2,1,0.952973,0.265185,0.952984,0.000834,0.265185,0.003796,2678.2,2298,3350,GPU,0.063944,7,11.114989,0.041169,39,128
3,5,0.952837,0.269200,0.952848,0.000917,0.269200,0.003271,5341.2,4546,6368,GPU,0.058049,5,1.394035,1.368466,18,254
4,3,0.952728,0.273093,0.952735,0.000834,0.273093,0.002425,7992.0,7983,7998,GPU,0.018945,5,3.476649,0.912140,32,254
5,4,0.952465,0.272676,0.952474,0.000930,0.272676,0.002532,7974.0,7925,7999,GPU,0.040437,4,7.896187,0.341048,3,254
6,2,0.952054,0.275567,0.952061,0.000893,0.275567,0.002413,7995.4,7990,7999,GPU,0.020337,4,2.814509,1.049513,18,254


In [13]:
# ============================================================
# Save artifacts
# ============================================================

print_section("Save Artifacts")

# Full trial table
trials_path = CFG.OUTDIR / "optuna_trials_045.csv"
trials_df.to_csv(trials_path, index=False)

# Best params only
best_params_path = CFG.OUTDIR / "best_params_045.json"
with open(best_params_path, "w") as f:
    json.dump(best_params, f, indent=2)

# Study summary
study_summary_path = CFG.OUTDIR / "optuna_study_summary_045.json"
with open(study_summary_path, "w") as f:
    json.dump(summary, f, indent=2)

# Pickled study for later inspection
study_pkl_path = CFG.OUTDIR / "optuna_study_045.pkl"
with open(study_pkl_path, "wb") as f:
    pickle.dump(study, f)

# Feature columns used in the search
feature_df = pd.DataFrame({
    "feature": common_features,
    "is_cat": [c in cat_cols for c in common_features],
    "is_count": [c.endswith("_count") for c in common_features],
    "is_freq": [c.endswith("_freq") for c in common_features],
    "is_digit": [
        ("_int_digit_" in c) or ("_dec_digit_" in c) or ("_sig_" in c)
        for c in common_features
    ],
    "is_string_precision": [
        c in ["RaceProgress_str", "EstimatedTotalLaps_str", "TyreAgeRatio_str"]
        for c in common_features
    ],
    "is_group_stat": [
        ("_mean_by_" in c) or ("_std_by_" in c) or ("_diff_mean_by_" in c)
        for c in common_features
    ],
    "is_race_year_group_stat": [
        is_race_year_groupstat_feature(c)
        for c in common_features
    ],
    "contains_year": ["Year" in c for c in common_features],
    "contains_race_year": ["Race_Year" in c for c in common_features],
    "is_shared001v2_042": [
        c.endswith("_floor_cat_042")
        or c in [
            "PitStop_cat_",
            "Race_Year_",
            "Race_Compound_",
            "RaceProgress_200_quantile_bin_",
            "LapTime (s)_7_quantile_bin_",
        ]
        or c.startswith("_PitStop_cat_")
        or c.startswith("_Race_Year__")
        or c.startswith("_Race_Compound__")
        or c.startswith("_RaceProgress_200_quantile_bin__")
        or c.startswith("_LapTime_s_7_quantile_bin__")
        for c in common_features
    ],
})
feature_df.to_csv(CFG.OUTDIR / "feature_columns.csv", index=False)

pd.DataFrame({
    "dropped_feature": dropped_groupstat_features_015,
    "reason": [
        "015/042 policy: remove Race_Year group-stat feature"
        for _ in dropped_groupstat_features_015
    ],
}).to_csv(CFG.OUTDIR / "dropped_features_015.csv", index=False)

memo = f"""experiment:
  id: {CFG.EXP_ID}
  competition: {CFG.COMPETITION}
  metric: {CFG.METRIC}
  created_at: 2026-05-20
  type: optuna_search
  status: completed

objective:
  summary: >
    042 CatBoost shared001v2 features構成を固定し、
    CatBoostの主要ハイパラだけをOptunaで探索した。
    特徴量追加やstack変更は行っていない。

base:
  experiment: exp_20260520_042_cat_shared001v2_features_min
  model: CatBoostClassifier
  baseline_cv_auc: {CFG.BASELINE_042_CV_AUC}
  baseline_public_lb: {CFG.BASELINE_042_PUBLIC_LB}

fixed:
  feature_set: same_as_042
  folds: same_as_042
  n_splits: {CFG.N_SPLITS}
  original_rows: same_as_042
  use_original: {str(CFG.USE_ORIGINAL).lower()}
  target: {CFG.TARGET}
  metric: AUC
  forbidden:
    - Normalized_TyreLife
    - Normalized_TyreLife proxy
    - future endpoint proxy
    - pseudo label
    - new feature engineering
    - 039 LOG features

search_space:
  learning_rate:
    type: float_log
    low: 0.015
    high: 0.08
  depth:
    type: int
    low: 4
    high: 8
  l2_leaf_reg:
    type: float_log
    low: 1.0
    high: 30.0
  random_strength:
    type: float
    low: 0.0
    high: 2.0
  min_data_in_leaf:
    type: int
    low: 1
    high: 40
  border_count:
    type: categorical
    choices: [128, 254]

fixed_catboost_params:
  loss_function: Logloss
  eval_metric: AUC
  task_type: {CFG.TASK_TYPE}
  devices: "{CFG.DEVICES}"
  bootstrap_type: {CFG.BOOTSTRAP_TYPE}
  subsample: {CFG.SUBSAMPLE}
  iterations: {CFG.ITERATIONS}
  early_stopping_rounds: {CFG.EARLY_STOPPING_ROUNDS}
  auto_class_weights: {CFG.AUTO_CLASS_WEIGHTS}

optuna:
  direction: {CFG.OPTUNA_DIRECTION}
  n_trials_requested: {CFG.N_TRIALS}
  n_trials_completed: {len(study.trials)}
  timeout_sec: {CFG.TIMEOUT_SEC}
  sampler: TPESampler
  seed: {CFG.SEED}

result:
  best_trial_number: {int(best_trial.number)}
  best_cv_auc: {best_cv_auc}
  delta_vs_042_cv: {best_cv_auc - CFG.BASELINE_042_CV_AUC}
  best_params:
{chr(10).join([f"    {k}: {json.dumps(v)}" for k, v in best_params.items()])}

outputs:
  best_params: best_params_045.json
  trials: optuna_trials_045.csv
  study_summary: optuna_study_summary_045.json
  study_pickle: optuna_study_045.pkl
  feature_columns: feature_columns.csv
  memo_draft: memo_draft.yml

success_criteria:
  best_cv_auc_must_exceed: {CFG.BASELINE_042_CV_AUC}
  strong_if_cv_auc_at_least: 0.95335

judgement:
  decision: pending
  note: >
    045は探索段階であり、正式OOF/pred/submissionは作成していない。
    best_cv_aucが042を上回る場合、046でbest_paramsを適用した正式refitを行う。
"""

memo_path = CFG.OUTDIR / "memo_draft.yml"
with open(memo_path, "w") as f:
    f.write(memo)

print("Saved:")
print(" -", trials_path)
print(" -", best_params_path)
print(" -", study_summary_path)
print(" -", study_pkl_path)
print(" -", CFG.OUTDIR / "feature_columns.csv")
print(" -", CFG.OUTDIR / "dropped_features_015.csv")
print(" -", memo_path)

print("\nBest params for 046:")
print(json.dumps(best_params, indent=2))



Save Artifacts
Saved:
 - /kaggle/working/exp_20260520_045_cat_shared001v2_features_optuna_search_min/optuna_trials_045.csv
 - /kaggle/working/exp_20260520_045_cat_shared001v2_features_optuna_search_min/best_params_045.json
 - /kaggle/working/exp_20260520_045_cat_shared001v2_features_optuna_search_min/optuna_study_summary_045.json
 - /kaggle/working/exp_20260520_045_cat_shared001v2_features_optuna_search_min/optuna_study_045.pkl
 - /kaggle/working/exp_20260520_045_cat_shared001v2_features_optuna_search_min/feature_columns.csv
 - /kaggle/working/exp_20260520_045_cat_shared001v2_features_optuna_search_min/dropped_features_015.csv
 - /kaggle/working/exp_20260520_045_cat_shared001v2_features_optuna_search_min/memo_draft.yml

Best params for 046:
{
  "learning_rate": 0.015888820918127773,
  "depth": 8,
  "l2_leaf_reg": 2.411289811529197,
  "random_strength": 1.325044568707964,
  "min_data_in_leaf": 13,
  "border_count": 254
}
